In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 14 — A CAÇA AOS HIPERPARÂMETROS: GRID SEARCH E RANDOM SEARCH

> "Um bom artesão conhece suas ferramentas. Um mestre sabe como afiá-las. A otimização de hiperparâmetros é a arte de afiar nossas ferramentas para a tarefa em mãos."
> — Um Engenheiro de Machine Learning experiente

Meu *pipeline* era uma máquina bem construída, mas toda máquina tem botões de ajuste. No meu, esses botões eram os **hiperparâmetros**: o `k` da seleção, o `C` e o `gamma` do SVM. Eu os deixara em valores "razoáveis", sabendo que estava deixando desempenho na mesa.

Otimizá-los à mão seria um pesadelo combinatório: testar 4 valores de `k`, 4 de `C` e 4 de `gamma` já são 64 combinações, cada uma exigindo uma validação cruzada de 5 *folds* — 320 treinos. Tarefa para uma máquina, não para um humano. Eu precisava de uma busca sistemática.

## Otimização de Hiperparâmetros

**Parâmetros** são aprendidos dos dados (ex.: os coeficientes β). **Hiperparâmetros** são configurações que definimos *antes* do treino. Duas estratégias de busca:

**Grid Search** — testa exaustivamente **todas** as combinações de uma grade. Garante encontrar a melhor da grade, mas o custo cresce exponencialmente.

**Random Search** — amostra aleatoriamente um número fixo de combinações. Costuma achar uma solução tão boa (ou melhor) em fração do tempo.

> **📘 PARA SABER — Por que Random Search costuma vencer**
>
> Bergstra e Bengio (2012) mostraram que, na prática, poucos hiperparâmetros realmente importam. O Grid Search gasta tempo variando os irrelevantes; o Random Search, amostrando ao acaso, tem mais chance de acertar os valores dos que importam. Para grades pequenas como a nossa, o Grid Search ainda é viável — e o usaremos aqui.

## Otimizando com GridSearchCV

O `GridSearchCV` combina busca em grade com validação cruzada. Para ajustar hiperparâmetros dentro de um *pipeline*, usamos a sintaxe `nome_do_passo__nome_do_hiperparametro`.

#### Código 14.1: GridSearchCV Sobre o Pipeline


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from oncoclassify_utils import X_train, y_train

def info_mutua(X, y):
    return mutual_info_classif(X, y, random_state=42)

pipeline = Pipeline([
    ("selecao", SelectKBest(score_func=info_mutua)),
    ("escala",  StandardScaler()),
    ("svm",     SVC(kernel="rbf", random_state=42)),
])

grade = {
    "selecao__k":  [10, 15, 20, 25],       # quantas features manter
    "svm__C":      [0.1, 1, 10, 100],      # regularizacao
    "svm__gamma":  ["scale", 0.1, 0.01, 0.001],
}

busca = GridSearchCV(pipeline, grade, cv=5, scoring="accuracy", n_jobs=-1)
busca.fit(X_train, y_train)

print("Melhor combinacao:", busca.best_params_)
print(f"Melhor acuracia (CV): {busca.best_score_:.4f}")

Melhor combinacao: {'selecao__k': 25, 'svm__C': 10, 'svm__gamma': 0.01}
Melhor acuracia (CV): 0.9771


A caça deu certo. Entre 64 combinações, a melhor mantém **25 *features***, usa **C=10** (fronteira mais flexível) e **gamma=0.01** (influência de médio alcance), alcançando **97,71%** de acurácia — acima dos 97,50% do SVM-RBF com parâmetros padrão. O objeto `busca.best_estimator_` já é o *pipeline* completo, treinado com essa combinação ótima. É, até agora, nosso melhor e mais robusto candidato a modelo final.

> **📌 CHECKLIST DO CAPÍTULO 14**
>
> ✅ Entende a diferença entre **parâmetros** e **hiperparâmetros**.
>
> ✅ Conhece **Grid Search** (exaustivo) e **Random Search** (amostrado, mais eficiente).
>
> ✅ Sabe usar a sintaxe `passo__hiperparametro` num *pipeline*.
>
> ✅ Executou o Código 14.1 e obteve a melhor combinação (**k=25, C=10, gamma=0.01**; acurácia 97,71%).
>
> **⚠️ CRÍTICO:** A otimização é cara, mas é uma das etapas que mais agregam. Sempre reserve tempo para afinar o modelo final — de preferência com `RandomizedSearchCV` quando a grade for grande.

Eu olhava os resultados com satisfação: a máquina fora afinada. Mas a reflexão me perturbava. **97,71% de acurácia.** O número era alto, mas a palavra "acurácia" parecia incompleta. Se o modelo errasse, *como* erraria? Classificaria um tumor maligno como benigno — um falso negativo, o pior erro possível? A acurácia, sozinha, não me contava essa história. Eu precisava mergulhar nos erros do modelo. Precisava ir além da acurácia.

**PARTE V — AVALIAÇÃO E CALIBRAÇÃO**
